# Count and measure Qwen3-8B

In [ ]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("Qwen/Qwen3-8B")
config.num_hidden_layers, config.hidden_size, config.num_key_value_heads

In [ ]:
V = config.vocab_size
d = config.hidden_size
L = config.num_hidden_layers
heads = config.num_attention_heads
kv_heads = config.num_key_value_heads
head_dim = config.head_dim
ff = config.intermediate_size

In [ ]:
attention = d * heads * head_dim + 2 * d * kv_heads * head_dim + heads * head_dim * d
feed_forward = 3 * d * ff
norms = 2 * d + 2 * head_dim
per_layer = attention + feed_forward + norms
total = 2 * V * d + L * per_layer + d
print(f"{total:,}")

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-8B", dtype=torch.bfloat16, device_map="cuda"
)

In [ ]:
print(f"{model.num_parameters():,}")

In [ ]:
GB = 1024 ** 3
print(f"{model.get_memory_footprint() / GB:.2f} GB")
print(f"{torch.cuda.memory_allocated() / GB:.2f} GB")

In [ ]:
for n in [2048, 4096, 8192]:
    ids = torch.randint(0, V, (1, n), device="cuda")
    with torch.no_grad():
        out = model(ids, use_cache=True, logits_to_keep=1)
    cache = out.past_key_values
    kv = sum(layer.keys.nbytes + layer.values.nbytes for layer in cache.layers)
    print(n, kv / 1024 ** 2, "MB")

In [ ]:
cache.layers[0].keys.shape

In [ ]:
2 * L * kv_heads * head_dim * 8192 * 2 / 1024 ** 2

In [ ]:
kv / 8192 / 1024

In [ ]:
print(f"{torch.cuda.memory_allocated() / GB:.2f} GB")